In [1]:
import pandas as pd
import requests
import os
import plotly.graph_objects as go
from calculadora_inflacao import CalculadoraFatorInflacao

# APOGESP
## Cálculo de impacto orçamentário das propostas reajuste salarial para os APPGGS

Esse notebook realiza a estimativa de impacto orçamentário das propostas de reajuste salarial feitas pela APOGESP para a campanha salarial de 2026 na Prefeitura de São Paulo.

São consideradas duas propostas principais:
1. Reajuste inflacionário, que se subdivide em:
    1. Reajuste da tabela original da carreira considerando a inflação acumulada desde as primeiras nomeações (jun/2026)
    2. Reajuste da tabela da carreira considerando a inflação acumulada na gestão Ricardo Nunes e a tabela no momento de início da gestão (jun/2021)
2. Equiparação com a tabela dos AMCI (ou seja, unificação da estrutura remuneratória do quadro QPPGG)


Em ambos os casos, a estimativa segue a seguinte metodologia:

1. Identificação dos APPGGs em exercício (incluso cedidos) segundo os últimos dados disponíveis no portal de dados abertos (dezembro de 2025),
2. Atualização dos dados do item 1 considerando o acréscimo dos 30 APPGGs recém-nomeados no nível 1, criando dados sintéticos (novas linhas na tabela)
3. Cálculo do salário base atual de cada APPGG considerando a tabela atual e o nível em que ele se encontra,
4. Cálculo dos encargos e benefícios que tem como base o salário atual como contribuição previdenciária obrigatória, auxílio transporte, auxílio alimentação etc.
5. Cálculo do salário proposto considerando o nível em que o APPGG se encontra atualmente e a tabela atualizada proposta
6. Cálculo dos encargos citados no item 6,
7. Somatória dos salários e encargos para cada APPGG considerando a tabela atual e a tabela proposta,
8. Cálculo da diferença entre a situação atual e a situação proposta conforme item 7,
9. Somatória da diferença por nível da carreira APPGG (p. ex., para o nível 1 a proposta implica em X mil reais mensais para a Prefeitura para o nível 2, y mil reais etc.)
10. Simulação de progressão da carreira considerando 18 meses para a "subida" de nível e projeção do impacto orçamentário anual até o final da gestão (2028)

O diagrama na imagem abaixo representa as etapas da metodologia.

![Grafo da metodologia](mermaid_graph.svg)

## Dados servidores

Nessa seção pegamos os dados dos servidores ativos mais atualizados (dezembro de 2025) do portal de dados abertos. Em seguida, identificamos os APPGGs e adicionamos dados sintéticos referentes aos APPGGs recém-nomeados.

In [2]:
URL_SERVIDORES = 'https://dados.prefeitura.sp.gov.br/dataset/bf5df0f4-4fb0-4a5e-b013-07d098cc7b1c/resource/459da127-4642-4412-a3f1-762d42a165ff/download/verificado_ativos_03-06-2026_mai-2026.csv'
file_dados = 'servidores_ativos_maio.csv'

In [3]:
def download_dados_servidores(fname:str, url:str=URL_SERVIDORES)->str:

    if not os.path.exists(fname):
        print('Downloading data')
        with requests.get(url) as r:
            r.raise_for_status()
            content = r.content
            with open(fname, 'wb') as f:
                f.write(content)
            
            assert os.path.exists(fname)
            return fname
    return fname

In [4]:
file_dados = download_dados_servidores(file_dados)

In [5]:
df = pd.read_csv(file_dados, encoding='latin1', sep=';')

In [6]:
df.head()

,REGISTRO,VINCULO,NOME,CARGO_BASICO,REF_CARGO_BAS,SEGMENTO,GRUPO,SUBGRUPO,ESCOL_CARGO_BASICO,CARGO_COMISSAO,...,SECRET_SUBPREF,SIGLA,SETOR,DISTRITO,SUBPREFEITURA,ORGAO_EXT,SEXO,ANO_NASCIMENTO,RACA_COR,PCD
0,1145541,16,CLEUZA BORGES PEREIRA SILVA,ASSESSOR IV,CDA-4,NaN,QC,CARGO EM COMISSAO,NAO SE APLICA,NaN,...,SECRETARIA MUNICIPAL DE GESTAO,SEGES,ASSESSORIA JURIDICA,SE,SE,NaN,FEMININO,1949,PARDA,NAO
1,1160206,5,ADILSON DA SILVA,COORDENADOR II,CDA-6,NaN,QC,CARGO EM COMISSAO,SUPERIOR COMPLETO,NaN,...,SUBPREFEITURA SANTO AMARO,SUB-SA,COORDENADORIA DE PLANEJAMENTO E DESENVOLVIMENT...,SANTO AMARO,SANTO AMARO,NaN,MASCULINO,1949,BRANCA,NAO
2,1160478,2,JULIO DE CARVALHO,FISCAL DE POSTURAS MUNICIPAIS NIVEL IV,QFPM15,NaN,QFPM,SUPERIOR,SUPERIOR COMPLETO,NaN,...,SUBPREFEITURA SANTANA/TUCURUVI,SUB-ST,SUPERVISÇO TECNICA DE FISCALIZACAO,TUCURUVI,SANTANA/TUCURUVI,NaN,MASCULINO,1951,BRANCA,NAO
3,1161181,9,NEUSA PEDRAO NASSIR,ASSESSOR V,CDA-5,NaN,QC,CARGO EM COMISSAO,NAO SE APLICA,NaN,...,SECRETARIA DO GOVERNO MUNICIPAL,SGM,ASSESSORIA JURIDICA,SE,SE,NaN,FEMININO,1947,BRANCA,NAO
4,1168185,4,MARIA HELENA DI VERNIERI CUPPARI,ASSISTENTE TECNICO EDUCACIONAL,QPE17A,NaN,QPE L. 14660/07 RGPS,CARGO EM COMISSAO,LICENCIATURA PLENA COMPLETA,NaN,...,SECRETARIA MUNICIPAL DE EDUCACAO,SME,DIRETORIA REGIONAL DE EDUCACAO PIRITUBA/JARAGUA,LAPA,LAPA,NaN,FEMININO,1942,BRANCA,NAO


In [7]:
cargos = df['REF_CARGO_BAS'].unique()

In [8]:
for cargo in cargos:
    if "appgg" in str(cargo).lower():
        print(cargo)

APPGG3
APPGG1
APPGG2
APPGG5
APPGG6
APPGG4


In [9]:
#filtrando para apenas servidores efetivos (ou seja, que possuem cargo base)
df_efetivos = df[df['REF_CARGO_BAS'].notna()]

In [10]:
appggs = df_efetivos[df_efetivos['REF_CARGO_BAS'].str.contains('APPGG')].reset_index(drop=True)

In [11]:
appggs.head()

,REGISTRO,VINCULO,NOME,CARGO_BASICO,REF_CARGO_BAS,SEGMENTO,GRUPO,SUBGRUPO,ESCOL_CARGO_BASICO,CARGO_COMISSAO,...,SECRET_SUBPREF,SIGLA,SETOR,DISTRITO,SUBPREFEITURA,ORGAO_EXT,SEXO,ANO_NASCIMENTO,RACA_COR,PCD
0,6394604,3,CLAUDIO AGUIAR ALMEIDA,ANALISTA POLITICAS PUBLICAS GESTAO GOVERNAMENT...,APPGG3,NaN,QPGG,SUPERIOR,SUPERIOR COMPLETO,NaN,...,SECRETARIA MUNICIPAL DE CULTURA E ECONOMIA CRI...,SMC,SECRETARIA MUNICIPAL DE CULTURA E ECONOMIA CRI...,SE,SE,NaN,MASCULINO,1963,PRETA,NAO
1,7575491,2,MAURICIO DA SILVA CORREIA,ANALISTA POLITICAS PUBLICAS GESTAO GOVERNAMENT...,APPGG1,NaN,QPGG,SUPERIOR,SUPERIOR COMPLETO,NaN,...,SECRETARIA MUNICIPAL DE GESTAO,SEGES,SECRETARIA MUNICIPAL DE GESTAO,SE,SE,PREFEITURA DO MUNICIPIO DE ITAJAI,MASCULINO,1986,BRANCA,NAO
2,7718543,6,MARCIA MIYUKI ISHIKAWA,ANALISTA POLITICAS PUBLICAS GESTAO GOVERNAMENT...,APPGG2,NaN,QPGG,SUPERIOR,SUPERIOR COMPLETO,NaN,...,SECRETARIA MUNICIPAL DE HABITACAO,SEHAB,DEPARTAMENTO DE PLANEJAMENTO HABITACIONAL,SE,SE,NaN,FEMININO,1978,BRANCA,NAO
3,7794720,2,TIAGO ROSA MACHADO,ANALISTA POLITICAS PUBLICAS GESTAO GOVERNAMENT...,APPGG2,NaN,QPGG,SUPERIOR,SUPERIOR COMPLETO,ASSESSOR III,...,SECRETARIA MUNICIPAL DE ESPORTES E LAZER,SEME,SECRETARIA MUNICIPAL DE ESPORTES E LAZER,MOEMA,VILA MARIANA,NaN,MASCULINO,1983,BRANCA,NAO
4,7840501,2,THAIS ROBERTO DA SILVA,ANALISTA POLITICAS PUBLICAS GESTAO GOVERNAMENT...,APPGG5,NaN,QPGG,SUPERIOR,SUPERIOR COMPLETO,NaN,...,SECRETARIA MUNICIPAL DE DIREITOS HUMANOS E CID...,SMDHC,SECRETARIA MUNICIPAL DE DIREITOS HUMANOS E CID...,SE,SE,NaN,FEMININO,1977,PRETA,NAO


In [12]:
appggs = appggs[['REGISTRO', 'NOME', 'REF_CARGO_BAS', 'SIGLA', 'DATA_INICIO_EXERC']]

renomear_cols = {
    'REGISTRO' : 'rf',
    'NOME' : 'nome',
    'REF_CARGO_BAS' : 'cargo_base',
    'SIGLA' : 'secretaria_dez_2025',
    'DATA_INICIO_EXERC' : 'dt_inicio_exercicio'
}

appggs.rename(renomear_cols, axis=1, inplace=True)


In [13]:
appggs['nivel_carreira'] = appggs['cargo_base'].str.extract(r'(\d+)$').astype(int)

In [14]:
appggs.head()

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira
0,6394604,CLAUDIO AGUIAR ALMEIDA,APPGG3,SMC,01/10/2021,3
1,7575491,MAURICIO DA SILVA CORREIA,APPGG1,SEGES,03/11/2021,1
2,7718543,MARCIA MIYUKI ISHIKAWA,APPGG2,SEHAB,08/12/2021,2
3,7794720,TIAGO ROSA MACHADO,APPGG2,SEME,05/01/2022,2
4,7840501,THAIS ROBERTO DA SILVA,APPGG5,SMDHC,29/11/2017,5


In [15]:
appggs.shape

(194, 6)

In [16]:
#SEM RECEM NOMEADOS, MAS VOU DEIXAR O CODIGO PARA PODER COLOCAR DEPOIS SE FOR O CASO

QTD_RECEM_NOMEADOS=0
recem_nomeados = [{
            'rf' : str(i+1).zfill(7),
            'nome' : f'recem_nomeado_{i+1}',
            'cargo_base' : 'APPGG1',
            'secretaria_dez_2025' : 'NA',
            'nivel_carreira' : 1,
            'dt_inicio_exercicio' : '01/03/2026'
            }
            for i in range(QTD_RECEM_NOMEADOS)
            ]

recem_nomeados = pd.DataFrame(recem_nomeados)

In [17]:
recem_nomeados.head()

""


In [18]:
recem_nomeados.shape

(0, 0)

In [19]:
appggs = pd.concat([appggs, recem_nomeados])

In [20]:
appggs.shape

(194, 6)

In [21]:
#vou colocar como datetime porque precisamos calcular com base na data
appggs['dt_inicio_exercicio'] = pd.to_datetime(appggs['dt_inicio_exercicio'], format ="%d/%m/%Y")


#inclusive ja vou definir se a pessoa contribui para o regime proprio (IPREM) ou nao
appggs['contribui_rpps'] = appggs['dt_inicio_exercicio'].dt.year<2018

In [22]:
appggs.head()

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,contribui_rpps
0,6394604,CLAUDIO AGUIAR ALMEIDA,APPGG3,SMC,2021-10-01,3,False
1,7575491,MAURICIO DA SILVA CORREIA,APPGG1,SEGES,2021-11-03,1,False
2,7718543,MARCIA MIYUKI ISHIKAWA,APPGG2,SEHAB,2021-12-08,2,False
3,7794720,TIAGO ROSA MACHADO,APPGG2,SEME,2022-01-05,2,False
4,7840501,THAIS ROBERTO DA SILVA,APPGG5,SMDHC,2017-11-29,5,True


# Situaçao atual

Nessa seção calculamos a situação atual de dispêndio com a carreira de APPGGs por parte da Prefeitura, considerando apenas os vencimentos e os encargos que estão relacionados aos vencimentos, como contribuição previdenciaria.

link para tabela usada: https://clic.prefeitura.sp.gov.br/storage/uploads/2026/06/10/Tabelas%20de%20Vencimentos_05_2026%20Clic%202026.pdf

In [23]:
tabela_atual = {
    1 : 13815.84,
    2 : 15197.43,
    3 : 15577.36,
    4 : 15966.79,
    5 : 16365.96,
    6 : 16775.11,
    7 : 18788.14,
    8 : 19257.84,
    9 : 19739.29,
    10 : 20232.76,
    11 : 20738.60,
    12 : 22875.96,
    13 : 23447.85,
    14 : 24034.05,
    15 : 24634.90,
}

In [24]:
appggs_atual = appggs.copy(deep=True)

In [25]:
appggs_atual['vencimento'] = appggs_atual['nivel_carreira'].map(tabela_atual)

In [26]:
appggs_atual.head()

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,contribui_rpps,vencimento
0,6394604,CLAUDIO AGUIAR ALMEIDA,APPGG3,SMC,2021-10-01,3,False,15577.36
1,7575491,MAURICIO DA SILVA CORREIA,APPGG1,SEGES,2021-11-03,1,False,13815.84
2,7718543,MARCIA MIYUKI ISHIKAWA,APPGG2,SEHAB,2021-12-08,2,False,15197.43
3,7794720,TIAGO ROSA MACHADO,APPGG2,SEME,2022-01-05,2,False,15197.43
4,7840501,THAIS ROBERTO DA SILVA,APPGG5,SMDHC,2017-11-29,5,True,16365.96


### Terço adicional de férias e décimo terceiro

Nesse caso vamos dividir ambos por 12 são gastos anuais e nossa base é mensal.

In [27]:
def calcular_decimo_terceiro(row):

    return round(row['vencimento']/12, 2)

def calcular_terco_ferias(row):

    return round((row['vencimento']/3)/12, 2)

In [28]:
appggs_atual['decimo_terceiro'] = appggs_atual.apply(calcular_decimo_terceiro, axis=1)

appggs_atual['terco_ferias'] = appggs_atual.apply(calcular_terco_ferias, axis=1)

In [29]:
appggs_atual.head()

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,contribui_rpps,vencimento,decimo_terceiro,terco_ferias
0,6394604,CLAUDIO AGUIAR ALMEIDA,APPGG3,SMC,2021-10-01,3,False,15577.36,1298.11,432.70
1,7575491,MAURICIO DA SILVA CORREIA,APPGG1,SEGES,2021-11-03,1,False,13815.84,1151.32,383.77
2,7718543,MARCIA MIYUKI ISHIKAWA,APPGG2,SEHAB,2021-12-08,2,False,15197.43,1266.45,422.15
3,7794720,TIAGO ROSA MACHADO,APPGG2,SEME,2022-01-05,2,False,15197.43,1266.45,422.15
4,7840501,THAIS ROBERTO DA SILVA,APPGG5,SMDHC,2017-11-29,5,True,16365.96,1363.83,454.61


### Vale alimentação

O vale alimentação só é devido para quem ganha menos de 10 salários minimos. Como com o reajuste essa proporcao pode diminuir, ele deve entrar na conta apesar de ser um valor fixo.

In [30]:
def calcular_va(row):
    SALARIO_MINIMO = 1621

    valores = {
        3 : 750.84,
        5 : 500.56,
        6 : 375.42,
        7 : 250.26,
        #tem que repetir no ultimo caso
        10 : 250.26
    }

    #acima de 10 sm nao recebe
    if row['vencimento'] >= (SALARIO_MINIMO*10):
        return 0
    
    for key, value in valores.items():
        if row['vencimento'] <= SALARIO_MINIMO*key:
            return value

    raise ValueError(f'Valor do vencimento {row["vencimento"]} nao se encaixa em nenhum caso')

In [31]:
appggs_atual['vale_alimentacao'] = appggs_atual.apply(calcular_va, axis=1)

In [32]:
def vale_refeicao(row):
    #apesar de ser uma constante decidi adicionar para os valores absolutos ficarem mais corretos
    DIAS_MES = 22
    VALOR_DIARIO  = 31.28

    return DIAS_MES*VALOR_DIARIO


In [33]:
appggs_atual['vale_refeicao'] = appggs_atual.apply(vale_refeicao, axis=1)

In [34]:
appggs_atual.sample(3)

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,contribui_rpps,vencimento,decimo_terceiro,terco_ferias,vale_alimentacao,vale_refeicao
118,8914605,ANDRE RUZ NEVES,APPGG2,SEGES,2022-02-03,2,False,15197.43,1266.45,422.15,250.26,688.16
96,8894230,CASSIANA MONTESIAO DE SOUSA,APPGG3,SEGES,2021-09-23,3,False,15577.36,1298.11,432.70,250.26,688.16
122,8915237,ANDRE RONDON MATTANA,APPGG2,SMS,2022-02-04,2,False,15197.43,1266.45,422.15,250.26,688.16


### Contribuicao previdenciaria

Para calcular a contribuicao previdenciaria precisamos saber se a pessoa entrou antes de 2018 (ou seja, se é do IPREM) ou depois (se é do SAMPAPREV)

In [35]:
def valor_iprem(row):

    CONTRIBUICAO_IPREM = 0.28

    if not row['contribui_rpps']:
        return 0
    
    # o iprem nao considera o terço adicional de férias
    valor_base = row['vencimento'] + row['decimo_terceiro']
    
    return round(valor_base * CONTRIBUICAO_IPREM, 2)

In [36]:
appggs_atual['contribuicao_iprem'] = appggs_atual.apply(valor_iprem, axis=1)

In [37]:
appggs_atual.sample(4)

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,contribui_rpps,vencimento,decimo_terceiro,terco_ferias,vale_alimentacao,vale_refeicao,contribuicao_iprem
130,8915393,ANDREA GIANNELLA BANDEIRA,APPGG2,SEGES,2022-03-25,2,False,15197.43,1266.45,422.15,250.26,688.16,0.00
146,9411704,LUCAS COTOSCK LARA,APPGG1,SEPLAN,2024-07-15,1,False,13815.84,1151.32,383.77,250.26,688.16,0.00
22,8280266,MONICA DE AZEVEDO COSTA NOGARA,APPGG5,SMUL,2016-07-01,5,True,16365.96,1363.83,454.61,0.00,688.16,4964.34
24,8358842,RAISSA FONTELAS ROSADO GAMBI,APPGG5,SGM,2016-06-17,5,True,16365.96,1363.83,454.61,0.00,688.16,4964.34


In [38]:
def contribuicao_inss(row):

    ALIQUOTA_INSS = 0.21
    TETO_INSS = 8475.55

    if row['contribui_rpps']:
        return 0
    
    # o inss vai sobre todos os vencimentos, incluso terco adicional
    valor_base = row['vencimento'] + row['terco_ferias'] + row['decimo_terceiro']

    #só para validar o teto - no caso de APPGG da na mesma porque tá todo mundo acima, mas pra deixar a funcao correta
    if valor_base > TETO_INSS:
        valor_base = TETO_INSS
        
    return round(valor_base*ALIQUOTA_INSS, 2)
    


In [39]:
appggs_atual['contribuicao_inss'] = appggs_atual.apply(contribuicao_inss, axis=1)

In [40]:
appggs_atual.sample(4)

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,contribui_rpps,vencimento,decimo_terceiro,terco_ferias,vale_alimentacao,vale_refeicao,contribuicao_iprem,contribuicao_inss
129,8915385,PEDRO PAULO CARDOSO BARCELLOS FERREIRA,APPGG2,SEGES,2022-02-07,2,False,15197.43,1266.45,422.15,250.26,688.16,0.00,1779.87
41,8359059,DIEGO XAVIER LEITE,APPGG6,SEGES,2016-06-17,6,True,16775.11,1397.93,465.98,0.00,688.16,5088.45,0.00
51,8359393,NATHALIA LEONE MARCO,APPGG6,CGM,2016-07-05,6,True,16775.11,1397.93,465.98,0.00,688.16,5088.45,0.00
98,8894264,EDSON LEITE DE CAMPOS JUNIOR,APPGG3,SVMA,2021-10-08,3,False,15577.36,1298.11,432.70,250.26,688.16,0.00,1779.87


In [41]:
def previdencia_complementar(row):

    TETO_INSS = 8475.55
    ALIQUOTA_COMPLEMENTAR = 0.075
    if row['contribui_rpps']:
        return 0
    
    valor_base = row['vencimento'] + row['decimo_terceiro']
    #só contribui o que é acima do teto
    valor_base = valor_base - TETO_INSS

    return round(valor_base * ALIQUOTA_COMPLEMENTAR, 2)



In [42]:
appggs_atual['previdencia_complementar'] = appggs_atual.apply(previdencia_complementar, axis=1)

In [43]:
appggs_atual.sample(4)

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,contribui_rpps,vencimento,decimo_terceiro,terco_ferias,vale_alimentacao,vale_refeicao,contribuicao_iprem,contribuicao_inss,previdencia_complementar
145,9388796,JULIANA CRISTINE TOMONARI MATUZAKI HONDA,APPGG1,SME,2024-07-24,1,False,13815.84,1151.32,383.77,250.26,688.16,0.00,1779.87,486.87
31,8358931,LIA PALM,APPGG6,SEGES,2016-06-24,6,True,16775.11,1397.93,465.98,0.00,688.16,5088.45,0.00,0.00
3,7794720,TIAGO ROSA MACHADO,APPGG2,SEME,2022-01-05,2,False,15197.43,1266.45,422.15,250.26,688.16,0.00,1779.87,599.12
94,8894205,GABRIELLA FERREIRA LOPES DE OLIVEIRA,APPGG3,SGM,2021-10-26,3,False,15577.36,1298.11,432.70,250.26,688.16,0.00,1779.87,629.99


In [44]:
def calculo_ir_simples(base_calculo:float)->float:
    '''Calcula o IR de forma simplificada considerando que todos os APPGGs ganham já acima da última faixa
    então não precisa aplicar redutor nem isenção etc.'''

    ultima_faixa = 7350
    if base_calculo <= ultima_faixa:
        raise ValueError('Base de cálculo deve ser maior que a última faixa para o cálculo simplificado.')

    aliquota = 0.275
    deducao = 908.73
    return round((base_calculo*aliquota)-deducao, 2)

def irpf_recolhido_na_fonte(row):

    #o IRPF é sobre vencimento + 3o adicional de ferias + 13o, ou seja, sobre tudo que é pago mensalmente

    base_calculo = row['vencimento'] + row['terco_ferias'] + row['decimo_terceiro']
    #considerando que todos os APPGGs ganham acima da última faixa, posso usar o cálculo simplificado
    return calculo_ir_simples(base_calculo)

In [45]:
appggs_atual['irpf_na_fonte'] = appggs_atual.apply(irpf_recolhido_na_fonte, axis=1)
appggs_atual.sample(4)

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,contribui_rpps,vencimento,decimo_terceiro,terco_ferias,vale_alimentacao,vale_refeicao,contribuicao_iprem,contribuicao_inss,previdencia_complementar,irpf_na_fonte
173,9427996,VINICIUS ANAUE RODRIGUES PINTO,APPGG1,SEGES,2024-10-03,1,False,13815.84,1151.32,383.77,250.26,688.16,0.00,1779.87,486.87,3312.78
116,8907528,ADRIANO FRANCO FEITOSA,APPGG2,SEGES,2021-12-03,2,False,15197.43,1266.45,422.15,250.26,688.16,0.00,1779.87,599.12,3734.93
22,8280266,MONICA DE AZEVEDO COSTA NOGARA,APPGG5,SMUL,2016-07-01,5,True,16365.96,1363.83,454.61,0.00,688.16,4964.34,0.00,0.00,4091.98
47,8359172,GUSTAVO GUIMARAES DE CAMPOS RABELLO,APPGG6,SVMA,2016-06-29,6,True,16775.11,1397.93,465.98,0.00,688.16,5088.45,0.00,0.00,4217.00


## Valor total

Agora calculamos o valor total e salvamos os dados

In [46]:
def valor_total_prefeitura(row):

    vencimentos = row['vencimento'] + row['decimo_terceiro'] + row['terco_ferias']
    auxilios = row['vale_alimentacao'] + row['vale_refeicao']
    previdencia = row['contribuicao_iprem'] + row['contribuicao_inss'] + row['previdencia_complementar']

    custos_totais = vencimentos + auxilios + previdencia
    #agora tem que tirar o irpf porque a prefeitura recolhe na fonte e já vai pro tesouro
    irpf = row['irpf_na_fonte']
    return round(custos_totais - irpf, 2)

In [47]:
appggs_atual['valor_total_prefeitura'] = appggs_atual.apply(valor_total_prefeitura, axis=1)

In [48]:
appggs_atual.head()

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,contribui_rpps,vencimento,decimo_terceiro,terco_ferias,vale_alimentacao,vale_refeicao,contribuicao_iprem,contribuicao_inss,previdencia_complementar,irpf_na_fonte,valor_total_prefeitura
0,6394604,CLAUDIO AGUIAR ALMEIDA,APPGG3,SMC,2021-10-01,3,False,15577.36,1298.11,432.70,250.26,688.16,0.00,1779.87,629.99,3851.02,16805.43
1,7575491,MAURICIO DA SILVA CORREIA,APPGG1,SEGES,2021-11-03,1,False,13815.84,1151.32,383.77,250.26,688.16,0.00,1779.87,486.87,3312.78,15243.31
2,7718543,MARCIA MIYUKI ISHIKAWA,APPGG2,SEHAB,2021-12-08,2,False,15197.43,1266.45,422.15,250.26,688.16,0.00,1779.87,599.12,3734.93,16468.51
3,7794720,TIAGO ROSA MACHADO,APPGG2,SEME,2022-01-05,2,False,15197.43,1266.45,422.15,250.26,688.16,0.00,1779.87,599.12,3734.93,16468.51
4,7840501,THAIS ROBERTO DA SILVA,APPGG5,SMDHC,2017-11-29,5,True,16365.96,1363.83,454.61,0.00,688.16,4964.34,0.00,0.00,4091.98,19744.92


In [49]:
appggs_atual.to_csv('situacao_atual.csv', sep=';')

# Proposta reajuste inflação
### Reajuste tabela original

In [50]:
tabela_original = {
    1 : 9000.00,
    2 : 10080.00,
    3 : 10684.80,
    4 : 11325.89,
    5 : 12005.44,
    6 : 12725.77,
    7 : 13998.34,
    8 : 14698.26,
    9 : 15433.17,
    10 : 16204.83,
    11 : 17015.08,
    12 : 18716.58,
    13 : 19558.83,
    14 : 20438.98,
    15 : 21358.73
}

In [51]:
calculadora_inflacao = CalculadoraFatorInflacao('ipc-fipe')

In [52]:
# nesse momento o ultimo valor do IPCA disponível é para junho de 2026
fator_ipc = calculadora_inflacao('01/06/2016', '01/08/2026')

Dados obtidos com sucesso na URL: https://api.bcb.gov.br/dados/serie/bcdata.sgs.193/dados?formato=json&dataInicial=01%2F06%2F2016&dataFinal=01%2F08%2F2026


In [53]:
fator_ipc

1.5988636100912699

In [54]:
#calculo rapido para ver progressao
(12725.77-9000)*fator_ipc

5956.9980725697515

In [55]:
#Só vendo para efeitos de comparacao
fator_ipca = CalculadoraFatorInflacao('ipca')('01/06/2016', '01/03/2026')
fator_ipca

Dados obtidos com sucesso na URL: https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados?formato=json&dataInicial=01%2F06%2F2016&dataFinal=01%2F03%2F2026


1.613929066220535

In [56]:
tabela_atualizada = {nivel : round(valor*fator_ipc, 2) for nivel, valor in tabela_original.items()}
tabela_atualizada

{1: 14389.77,
 2: 16116.55,
 3: 17083.54,
 4: 18108.55,
 5: 19195.06,
 6: 20346.77,
 7: 22381.44,
 8: 23500.51,
 9: 24675.53,
 10: 25909.31,
 11: 27204.79,
 12: 29925.26,
 13: 31271.9,
 14: 32679.14,
 15: 34149.7}

In [57]:
if tabela_atualizada[1] > tabela_atual[1]:
    print('O reajuste pelo IPC é maior do que o reajuste concedido atualmente pela prefeitura, portanto podemos manter os valores, caso contrario se algum nível no reajuste atual fosse maior, teria que manter esse nível e não o da tabela atualizada pois não pode reduzir salário.')

O reajuste pelo IPC é maior do que o reajuste concedido atualmente pela prefeitura, portanto podemos manter os valores, caso contrario se algum nível no reajuste atual fosse maior, teria que manter esse nível e não o da tabela atualizada pois não pode reduzir salário.


In [58]:
proposta_ipc = appggs.copy(deep=True)

Agora vamos calcular rapidamente o que fizemos na seção anterior, qual seja:
1. o novo vencimento,
2. o décimo terceiro,
3. o terço adicional de férias,
4. o auxilio alimentacao,
5. as contribuicoes previdenciarias
6. o irpf na fonte
7. o valor total

In [59]:
#vencimento
proposta_ipc['vencimento'] = proposta_ipc['nivel_carreira'].map(tabela_atualizada)
#ferias e decimo terceiro
proposta_ipc['decimo_terceiro'] = proposta_ipc.apply(calcular_decimo_terceiro, axis=1)
proposta_ipc['terco_ferias'] = proposta_ipc.apply(calcular_terco_ferias, axis=1)
#vale alimentacao
proposta_ipc['vale_alimentacao'] = proposta_ipc.apply(calcular_va, axis=1)
proposta_ipc['vale_refeicao'] = proposta_ipc.apply(vale_refeicao, axis=1)
#previdencia
proposta_ipc['contribuicao_iprem'] = proposta_ipc.apply(valor_iprem, axis=1)
proposta_ipc['contribuicao_inss'] = proposta_ipc.apply(contribuicao_inss, axis=1)
proposta_ipc['previdencia_complementar'] = proposta_ipc.apply(previdencia_complementar, axis=1)
#irpf
proposta_ipc['irpf_na_fonte'] = proposta_ipc.apply(irpf_recolhido_na_fonte, axis=1)
#VALOR TOTAL PROPOSTA
proposta_ipc['valor_total_prefeitura'] = proposta_ipc.apply(valor_total_prefeitura, axis=1)

In [60]:
proposta_ipc.head()

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,contribui_rpps,vencimento,decimo_terceiro,terco_ferias,vale_alimentacao,vale_refeicao,contribuicao_iprem,contribuicao_inss,previdencia_complementar,irpf_na_fonte,valor_total_prefeitura
0,6394604,CLAUDIO AGUIAR ALMEIDA,APPGG3,SMC,2021-10-01,3,False,17083.54,1423.63,474.54,0.00,688.16,0.0,1779.87,752.37,4311.24,17890.87
1,7575491,MAURICIO DA SILVA CORREIA,APPGG1,SEGES,2021-11-03,1,False,14389.77,1199.15,399.72,250.26,688.16,0.0,1779.87,533.50,3488.15,15752.28
2,7718543,MARCIA MIYUKI ISHIKAWA,APPGG2,SEHAB,2021-12-08,2,False,16116.55,1343.05,447.68,250.26,688.16,0.0,1779.87,673.80,4015.77,17283.60
3,7794720,TIAGO ROSA MACHADO,APPGG2,SEME,2022-01-05,2,False,16116.55,1343.05,447.68,250.26,688.16,0.0,1779.87,673.80,4015.77,17283.60
4,7840501,THAIS ROBERTO DA SILVA,APPGG5,SMDHC,2017-11-29,5,True,19195.06,1599.59,533.20,0.00,688.16,5822.5,0.00,0.00,4956.43,22882.08


In [61]:
proposta_ipc.to_csv('proposta_ipc.csv', sep=';')

In [62]:
tabela_atual

{1: 13815.84,
 2: 15197.43,
 3: 15577.36,
 4: 15966.79,
 5: 16365.96,
 6: 16775.11,
 7: 18788.14,
 8: 19257.84,
 9: 19739.29,
 10: 20232.76,
 11: 20738.6,
 12: 22875.96,
 13: 23447.85,
 14: 24034.05,
 15: 24634.9}

In [63]:
tabela_atualizada

{1: 14389.77,
 2: 16116.55,
 3: 17083.54,
 4: 18108.55,
 5: 19195.06,
 6: 20346.77,
 7: 22381.44,
 8: 23500.51,
 9: 24675.53,
 10: 25909.31,
 11: 27204.79,
 12: 29925.26,
 13: 31271.9,
 14: 32679.14,
 15: 34149.7}

In [64]:
def tabela_to_pandas(tabela:dict, tabela_name:str):

    dados = []
    for nivel, val in tabela.items():
        dados.append([tabela_name, nivel, val])

    return pd.DataFrame(dados, columns = ['nome_tabela', 'nivel', 'vencimento'])

In [65]:
tabelas = {
    'atual' : tabela_atual,
    'original' : tabela_original,
    'original_atualizada_ipc' : tabela_atualizada,
}

In [66]:
dfs_tabelas = [tabela_to_pandas(tabela, tabela_name) for tabela_name, tabela in tabelas.items()]
df_tabelas = pd.concat(dfs_tabelas)

df_tabelas

,nome_tabela,nivel,vencimento
0,atual,1,13815.84
1,atual,2,15197.43
2,atual,3,15577.36
3,atual,4,15966.79
4,atual,5,16365.96
5,atual,6,16775.11
6,atual,7,18788.14
7,atual,8,19257.84
8,atual,9,19739.29
9,atual,10,20232.76


In [67]:
df_tabelas.to_csv('tabelas_vencimentos_utilizadas_aug_2026.csv', sep=';')

# Agregações

Agora fazemos as agregações para analisar o impacto orçamentário

In [68]:
def total_mensal_por_nivel(servidores, nome_proposta:str):

    df = servidores.groupby('nivel_carreira')['valor_total_prefeitura'].sum().reset_index()
    df['id_proposta'] = nome_proposta
    return df

In [69]:
total_atual = total_mensal_por_nivel(appggs_atual, 'situacao_atual')
total_atual

,nivel_carreira,valor_total_prefeitura,id_proposta
0,1,1190902.24,situacao_atual
1,2,592866.36,situacao_atual
2,3,439256.45,situacao_atual
3,4,19552.55,situacao_atual
4,5,293683.40,situacao_atual
5,6,767547.94,situacao_atual


In [70]:
mensal_atual = total_atual['valor_total_prefeitura'].sum()
mensal_atual

np.float64(3303808.94)

In [71]:
anual_atual = mensal_atual*12
anual_atual

np.float64(39645707.28)

In [72]:
total_ipc = total_mensal_por_nivel(proposta_ipc, 'reajuste_ipc')
total_ipc

,nivel_carreira,valor_total_prefeitura,id_proposta
0,1,1230729.37,reajuste_ipc
1,2,622209.60,reajuste_ipc
2,3,467812.39,reajuste_ipc
3,4,21677.27,reajuste_ipc
4,5,340112.50,reajuste_ipc
5,6,918049.22,reajuste_ipc


In [73]:
total_ipc_mensal = total_ipc['valor_total_prefeitura'].sum()
total_ipc_mensal

np.float64(3600590.3500000006)

In [74]:
total_ipc_anual = total_ipc_mensal*12
total_ipc_anual

np.float64(43207084.2)

In [75]:
total_ipc_anual-anual_atual

np.float64(3561376.920000002)

In [76]:
def impacto_mensal(situacao_atual, proposta):

    join = pd.merge(situacao_atual, proposta, on='nivel_carreira', suffixes=('_atual', '_proposta'))
    join['impacto_mensal'] = join['valor_total_prefeitura_proposta'] - join['valor_total_prefeitura_atual']

    return join



In [77]:
impacto_mensal_ipc = impacto_mensal(total_atual, total_ipc)


In [78]:
impacto_mensal_ipc.to_csv('impacto_mensal_ipc_aug_2026.csv', sep=';')
impacto_mensal_ipc

,nivel_carreira,valor_total_prefeitura_atual,id_proposta_atual,valor_total_prefeitura_proposta,id_proposta_proposta,impacto_mensal
0,1,1190902.24,situacao_atual,1230729.37,reajuste_ipc,39827.13
1,2,592866.36,situacao_atual,622209.60,reajuste_ipc,29343.24
2,3,439256.45,situacao_atual,467812.39,reajuste_ipc,28555.94
3,4,19552.55,situacao_atual,21677.27,reajuste_ipc,2124.72
4,5,293683.40,situacao_atual,340112.50,reajuste_ipc,46429.10
5,6,767547.94,situacao_atual,918049.22,reajuste_ipc,150501.28


In [79]:
impacto_anual = round(impacto_mensal_ipc['impacto_mensal'].sum()*12, 2)

In [80]:
impacto_anual

np.float64(3561376.92)

In [81]:
custo_anual_ipc = round(total_ipc['valor_total_prefeitura'].sum()*12, 2)

In [82]:
custo_anual_ipc

np.float64(43207084.2)

In [83]:
impactos_mensais = {'icp_fipe' : impacto_mensal_ipc}

In [84]:

def gerar_grafico_niveis_df(df, col_valor:str, titulo:str, nome_arquivo="grafico_niveis.png"):
    
    col_nivel = 'nivel_carreira'
    
    cores = ["steelblue"] * len(df)
    indice_maximo = df[col_valor].idxmax()
    cores[indice_maximo] = "crimson"

    # Formatação para o texto sobre as barras (R$ 1.234,56)
    texto_formatado = df[col_valor].apply(
        lambda x: f"R$ {x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
    )

    fig = go.Figure(data=[
        go.Bar(
            x=df[col_nivel],
            y=df[col_valor],
            text=texto_formatado,
            textposition="outside",
            marker_color=cores
        )
    ])

    fig.update_layout(
        title=dict(
            text=titulo,
            x=0.5,
            xanchor="center",
            font=dict(size=22)
        ),
        # Define os separadores globalmente: o primeiro é o decimal, o segundo é o de milhar
        separators=",.",
        plot_bgcolor="white",
        width=1200,
        height=600,
        xaxis=dict(
            title="Níveis de Carreira",
            tickangle=-45,
            categoryorder="array",
            categoryarray=df[col_nivel].tolist()
        ),
        yaxis=dict(
            title="Valores em R$",
            showgrid=True,
            gridcolor="lightgrey",
            # Formata os números do eixo Y com separador de milhar e 2 casas decimais
            tickformat=",2f"
        ),
        margin=dict(l=50, r=50, t=100, b=120)
    )

    fig.write_image(nome_arquivo)

    return fig

In [85]:
import kaleido

# Baixa ou localiza o Chrome automaticamente
kaleido.get_chrome()

<coroutine object get_chrome at 0x7271d3dfba70>

In [86]:

for nome, df in impactos_mensais.items():
    titulo = f"Custo total por nível: {nome}"
    nome_arquivo = f"grafico_niveis_{nome.replace(' ', '_').lower()}.png"
    gerar_grafico_niveis_df(df, "valor_total_prefeitura_proposta", titulo, nome_arquivo)

In [87]:
for proposta, df in impactos_mensais.items():
    valor_total = df['valor_total_prefeitura_proposta'].sum()*12
    valor_formatado = f"R$ {valor_total:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
    print(f"Valor total anual: {valor_formatado} - {proposta}")

Valor total anual: R$ 43.207.084,20 - icp_fipe


In [88]:
for proposta, df in impactos_mensais.items():
    try:
        valor_total = df['impacto_mensal'].sum()*12
        valor_formatado = f"R$ {valor_total:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
        print(f"Impacot total anual: {valor_formatado} - {proposta}")
    except KeyError:
        print(proposta, "não possui impacto")

Impacot total anual: R$ 3.561.376,92 - icp_fipe


In [89]:
niveis: DataFrame = pd.read_csv('tabelas_vencimentos_utilizadas.csv', sep=';', index_col=0)

In [90]:
niveis = niveis[niveis['nome_tabela']!='original_atualizada_ipc_nunes'].reset_index(drop=True)

In [91]:
niveis

,nome_tabela,nivel,vencimento
0,atual,1,13671.75
1,atual,2,15038.93
2,atual,3,15414.90
3,atual,4,15800.26
4,atual,5,16195.27
5,atual,6,16600.15
6,atual,7,18592.19
7,atual,8,19056.99
8,atual,9,19533.41
9,atual,10,20021.73


In [92]:
import plotly.graph_objects as go
import pandas as pd

def exportar_graficos_comparativos(df, col_valor="vencimento"):
    col_nivel = "nivel"
    col_tabela = "nome_tabela"
    
    # Identifica as tabelas que serão comparadas com a 'atual'
    tabelas_extras = [t for t in df[col_tabela].unique() if t != "atual"]
    
    for tabela in tabelas_extras:
        # Filtra apenas o par necessário
        df_par = df[df[col_tabela].isin(["atual", tabela])]
        
        fig = go.Figure()

        # Adiciona as barras para 'atual' e para a 'tabela' da vez
        for nome in ["atual", tabela]:
            df_sub = df_par[df_par[col_tabela] == nome]
            
            # Formatação Real (sem centavos para evitar sobreposição de texto)
            texto = df_sub[col_valor].apply(
                lambda x: f"R$ {x:,.0f}".replace(",", "X").replace(".", ",").replace("X", ".")
            )

            fig.add_trace(go.Bar(
                x=df_sub[col_nivel],
                y=df_sub[col_valor],
                name=nome.replace("_", " ").title(),
                text=texto,
                textposition="outside",
                marker_color="steelblue" if nome == "atual" else "crimson"
            ))

        fig.update_layout(
            title=dict(
                text=f"Comparativo: Atual vs {tabela.replace('_', ' ').title()}",
                x=0.5,
                font=dict(size=22)
            ),
            barmode="group",
            separators=",.",
            plot_bgcolor="white",
            width=1200,
            height=600,
            xaxis=dict(title="Nível de Carreira", type="category"),
            yaxis=dict(
                title="Vencimento (R$)",
                showgrid=True,
                gridcolor="lightgrey",
                tickformat=",0f"
            ),
            legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
            margin=dict(l=50, r=50, t=100, b=50)
        )

        # Salva cada gráfico com o nome da tabela correspondente
        nome_arquivo = f"comparativo_atual_vs_{tabela}.png"
        fig.write_image(nome_arquivo, engine="kaleido")
        print(f"Arquivo salvo: {nome_arquivo}")

# Exemplo de uso:
# exportar_graficos_comparativos(df)

In [93]:
exportar_graficos_comparativos(niveis)

/tmp/ipykernel_214684/3595305641.py:59: DeprecationWarning: 
Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.

  fig.write_image(nome_arquivo, engine="kaleido")


Arquivo salvo: comparativo_atual_vs_original.png


/tmp/ipykernel_214684/3595305641.py:59: DeprecationWarning: 
Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.

  fig.write_image(nome_arquivo, engine="kaleido")


Arquivo salvo: comparativo_atual_vs_amci.png


/tmp/ipykernel_214684/3595305641.py:59: DeprecationWarning: 
Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.

  fig.write_image(nome_arquivo, engine="kaleido")


Arquivo salvo: comparativo_atual_vs_original_atualizada_ipc.png


In [94]:
impacto: DataFrame = pd.read_csv('impactos_anuais.csv', sep=';', index_col=0)

In [95]:
appggs_atual.head()

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,contribui_rpps,vencimento,decimo_terceiro,terco_ferias,vale_alimentacao,vale_refeicao,contribuicao_iprem,contribuicao_inss,previdencia_complementar,irpf_na_fonte,valor_total_prefeitura
0,6394604,CLAUDIO AGUIAR ALMEIDA,APPGG3,SMC,2021-10-01,3,False,15577.36,1298.11,432.70,250.26,688.16,0.00,1779.87,629.99,3851.02,16805.43
1,7575491,MAURICIO DA SILVA CORREIA,APPGG1,SEGES,2021-11-03,1,False,13815.84,1151.32,383.77,250.26,688.16,0.00,1779.87,486.87,3312.78,15243.31
2,7718543,MARCIA MIYUKI ISHIKAWA,APPGG2,SEHAB,2021-12-08,2,False,15197.43,1266.45,422.15,250.26,688.16,0.00,1779.87,599.12,3734.93,16468.51
3,7794720,TIAGO ROSA MACHADO,APPGG2,SEME,2022-01-05,2,False,15197.43,1266.45,422.15,250.26,688.16,0.00,1779.87,599.12,3734.93,16468.51
4,7840501,THAIS ROBERTO DA SILVA,APPGG5,SMDHC,2017-11-29,5,True,16365.96,1363.83,454.61,0.00,688.16,4964.34,0.00,0.00,4091.98,19744.92


In [96]:
appggs_atual['nome'].str.startswith('recem_nomeado').sum()

np.int64(0)

In [97]:
qtd_por_nivel: DataFrame = appggs_atual.groupby('nivel_carreira').count()[['rf']].rename({'rf' : 'Quantidade'}, axis=1)

In [98]:
import plotly.graph_objects as go

fig = go.Figure(data=[
    go.Bar(
        x=qtd_por_nivel.index.astype(str),  # Convertendo o índice para string para melhor exibição
        y=qtd_por_nivel['Quantidade'],
        text=qtd_por_nivel['Quantidade'],
        textposition='outside',
        marker_color='steelblue'
    )
])

fig.update_layout(
    title=dict(
        text="Quantidade por Nível na Carreira",
        x=0.5,
        font=dict(size=22)
    ),
    xaxis=dict(
        title="Nível na Carreira",
        tickangle=0
    ),
    yaxis=dict(
        title="Quantidade",
        showgrid=True,
        gridcolor="lightgrey"
    ),
    plot_bgcolor="white",
    width=800,
    height=500,
    margin=dict(l=50, r=50, t=100, b=50)
)
fig.write_image('quantidade_pessoas_por_nivel.png', engine="kaleido")

fig.show()

/tmp/ipykernel_214684/2642592332.py:33: DeprecationWarning: 
Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.

  fig.write_image('quantidade_pessoas_por_nivel.png', engine="kaleido")
